# Frente C: Estratégia de decisão e política operacional

A Frente A não encontrou separação univariada e a Frente B não generalizou no OOT. Esta análise usa o score XGBoost existente para testar uma política de três zonas, medir o trade-off operacional e verificar estabilidade por safra e Instituição Financeira (IF).

O `saving` será tratado como proxy de exposição financiada em risco. Não há coluna de perda realizada, recuperação ou custo de investigação; portanto, nenhum valor será apresentado como saving observado.

## Como reconstruímos o score sem reintroduzir vazamento?

Carregamos as mesmas features da Frente B, usando histórico anterior, e aplicamos o modelo já treinado. Os limiares serão definidos somente pelo quantil do score na validação.

In [1]:
import sys
from pathlib import Path
import polars as pl

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src" / "politica_operacional_vrum.py").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))

from politica_operacional_vrum import (
    TARGET, aplicar_zonas, construir_features, carregar_base,
    pontuar_splits, resumo_zonas, estabilidade_por_if,
)

def reais(valor):
    return f"R$ {valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")

base = construir_features(carregar_base())
pontuado, limiar_investigar, limiar_bloquear = pontuar_splits(base)
pontuado = aplicar_zonas(pontuado, limiar_investigar, limiar_bloquear)
print("Propostas pontuadas:", pontuado.height)

Propostas pontuadas: 2129214


In [2]:
pl.DataFrame({
    "regra": ["investigar", "bloquear"],
    "quantil_validacao": [0.95, 0.99],
    "limiar_score": [limiar_investigar, limiar_bloquear],
})

regra,quantil_validacao,limiar_score
str,f64,f64
"""investigar""",0.95,0.54911
"""bloquear""",0.99,0.595998


**Resultado:** os limiares controlam capacidade operacional, não representam probabilidade calibrada de fraude. Como foram fixados na validação, podem ser aplicados ao OOT sem escolher corte olhando o futuro.

## Quanto cada zona captura e quanto encaminha para intervenção?

Aprovar não é intervenção. Captura de risco significa risco enviado para `INVESTIGAR` ou `BLOQUEAR`; falso positivo significa proposta legítima enviada para uma dessas zonas.

In [3]:
resumo = pl.concat([
    resumo_zonas(pontuado.filter(pl.col("safra") == safra), safra)
    for safra in ["treino", "validacao", "oot"]
])
resumo.select(["grupo", "zona_decisao", "n", "positivos", "taxa_risco_zona", "captura_risco", "falso_positivo_global", "exposicao_risco"])

grupo,zona_decisao,n,positivos,taxa_risco_zona,captura_risco,falso_positivo_global,exposicao_risco
str,str,u32,i64,f64,f64,f64,f64
"""treino""","""APROVAR""",710521,9989,0.014059,0.0,0.0,1.0707e9
"""treino""","""INVESTIGAR""",13618,540,0.039653,0.050661,0.018292,5.6506e7
"""treino""","""BLOQUEAR""",1470,130,0.088435,0.012196,0.001874,1.0859e7
"""validacao""","""APROVAR""",754910,10957,0.014514,0.0,0.0,1.2033e9
"""validacao""","""INVESTIGAR""",31786,438,0.01378,0.038064,0.040029,3.6798e7
"""validacao""","""BLOQUEAR""",7947,112,0.014093,0.009733,0.010005,7.3918e6
"""oot""","""APROVAR""",586104,8681,0.014811,0.0,0.0,9.4731e8
"""oot""","""INVESTIGAR""",18521,291,0.015712,0.032233,0.030387,2.2310e7
"""oot""","""BLOQUEAR""",4337,56,0.012912,0.006203,0.007136,3.6114e6


**Resultado:** a política concentra apenas pequena parcela do volume em intervenção, mas a captura OOT permanece baixa e a taxa de risco das zonas não supera de forma consistente a taxa-base. O corte não sustenta bloqueio automático.

## Qual seria o saving sob diferentes taxas de prevenção?

Usamos `valor_financiado` como exposição. Os cenários de 25%, 50% e 100% significam fração hipotética da exposição de risco evitada após intervenção; não são perdas estimadas.

In [4]:
resumo.filter(pl.col("grupo") == "oot").select([
    "zona_decisao", "n", "falsos_positivos", "exposicao_risco",
    "saving_proxy_25pct", "saving_proxy_50pct", "saving_proxy_100pct",
]).with_columns([
    pl.col(coluna).map_elements(reais, return_dtype=pl.String).alias(coluna)
    for coluna in ["exposicao_risco", "saving_proxy_25pct", "saving_proxy_50pct", "saving_proxy_100pct"]
])

zona_decisao,n,falsos_positivos,exposicao_risco,saving_proxy_25pct,saving_proxy_50pct,saving_proxy_100pct
str,u32,i64,str,str,str,str
"""APROVAR""",586104,0,"""R$ 947.310.982,92""","""R$ 0,00""","""R$ 0,00""","""R$ 0,00"""
"""INVESTIGAR""",18521,18230,"""R$ 22.309.983,52""","""R$ 5.577.495,88""","""R$ 11.154.991,76""","""R$ 22.309.983,52"""
"""BLOQUEAR""",4337,4281,"""R$ 3.611.419,41""","""R$ 902.854,85""","""R$ 1.805.709,71""","""R$ 3.611.419,41"""


**Resultado:** sem perda observada e sem custo de falso positivo, o número não permite decisão financeira. O artefato correto para a banca é uma análise de sensibilidade, condicionada à medição posterior de perda evitada e custo de mesa.

## A política permanece estável entre safras?

Comparamos taxa de intervenção, taxa de risco e exposição por `treino`, `validacao` e `oot`, mantendo os mesmos limiares.

In [5]:
por_safra = (pontuado.group_by("safra").agg([
    pl.len().alias("n"),
    pl.col(TARGET).sum().alias("positivos"),
    pl.col(TARGET).mean().alias("taxa_risco"),
    (pl.col("zona_decisao") != "APROVAR").mean().alias("taxa_intervencao"),
    (pl.col("zona_decisao") == "BLOQUEAR").mean().alias("taxa_bloqueio"),
]).sort("safra"))
por_safra

safra,n,positivos,taxa_risco,taxa_intervencao,taxa_bloqueio
str,u32,i64,f64,f64,f64
"""oot""",608962,9028,0.014825,0.037536,0.007122
"""treino""",725609,10659,0.01469,0.020794,0.002026
"""validacao""",794643,11507,0.014481,0.050001,0.010001


**Resultado:** diferenças entre safras mostram alteração de distribuição do score, enquanto o risco observado permanece perto da taxa-base. Estabilidade operacional não basta; é preciso estabilidade de captura e de valor evitado.

## A política é estável entre Instituições Financeiras?

Medimos volume, prevalência, distribuição das zonas e métricas de ranking por IF. IF com pouca amostra deve ser monitorada, não ranqueada como melhor ou pior.

In [6]:
estabilidade = estabilidade_por_if(pontuado)
estabilidade.group_by("safra").agg([
    pl.len().alias("ifs"),
    pl.col("n").min().alias("n_min"),
    pl.col("n").max().alias("n_max"),
    pl.col("taxa_investigar").min().alias("investigar_min"),
    pl.col("taxa_investigar").max().alias("investigar_max"),
    pl.col("taxa_bloquear").min().alias("bloquear_min"),
    pl.col("taxa_bloquear").max().alias("bloquear_max"),
    pl.col("auc_roc").min().alias("auc_min"),
    pl.col("auc_roc").max().alias("auc_max"),
]).sort("safra")

safra,ifs,n_min,n_max,investigar_min,investigar_max,bloquear_min,bloquear_max,auc_min,auc_max
str,u32,i64,i64,f64,f64,f64,f64,f64,f64
"""oot""",20,30111,30942,0.028827,0.032639,0.006561,0.007995,0.473491,0.511369
"""treino""",20,35948,36768,0.017538,0.020099,0.001236,0.002657,0.587359,0.635188
"""validacao""",20,39579,39989,0.038283,0.041811,0.009088,0.010609,0.478996,0.525065


**Resultado:** as taxas de encaminhamento variam entre IFs por distribuição de score, mas o AUC por IF permanece próximo de 0,5 no OOT. Não há base para política de bloqueio diferenciada por IF; primeiro, medir perdas reais e revisar o target.

## Qual política deve entrar em operação agora?

A política proposta é uma versão inicial para simulação e monitoramento, não uma autorização de recusa automática.

In [7]:
oot = pontuado.filter(pl.col("safra") == "oot")
intervencao = oot.filter(pl.col("zona_decisao") != "APROVAR")
intervencao.select([
    pl.len().alias("n_intervencao"),
    pl.col(TARGET).sum().alias("riscos_capturados"),
    pl.col(TARGET).mean().alias("taxa_risco_intervencao"),
]).with_columns([
    (pl.col("n_intervencao") / oot.height).alias("taxa_intervencao"),
    (pl.col("riscos_capturados") / oot[TARGET].sum()).alias("captura_risco"),
])

n_intervencao,riscos_capturados,taxa_risco_intervencao,taxa_intervencao,captura_risco
u32,i64,f64,f64,f64
22858,347,0.015181,0.037536,0.038436


**Decisão operacional:**

- `APROVAR`: fluxo direto apenas como decisão provisória, com monitoramento.
- `INVESTIGAR`: piloto em shadow mode ou mesa manual; medir tempo, custo e prevenção real.
- `BLOQUEAR`: manter desabilitado para recusa automática até existir target confiável e AUC/PR-AUC OOT acima da referência.

Monitorar mensalmente por IF e safra: prevalência, AUC, KS, PR-AUC, captura, falso positivo, taxa de intervenção, drift de score, exposição financiada e perda evitada confirmada.